# Аналитика отключений систем ЖКХ во Владивостоке

## Загрузка библиотек

In [1]:
import numpy as np
import pandas as pd
import sqlite3

import re

from IPython.display import display, HTML

from typing import Any, Tuple

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import classification_report
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import f1_score, make_scorer

import pickle

import warnings
from tqdm import tqdm

tqdm.pandas()
warnings.filterwarnings('ignore')

## Работа с данными

### Описание таблиц и столбцов

**blackouts** — таблица отключений систем ЖКХ 
- `id` — уникальный идентификатор  
- `start_date` — время начала отключения  
- `end_date` — время окончания отключения  
- `description` — описание отключения  
- `type` — тип отключения  
- `initiator_name` — инициатор отключения  
- `source` — источник информации  

**blackouts_buildings** — связь отключений со зданиями (многие-ко-многим)  
- `blackout_id` — ссылка на отключение  
- `building_id` — ссылка на здание  

**buildings** — справочник зданий  
- `id` — уникальный идентификатор  
- `street_id` — ссылка на улицу  
- `number` — номер дома  
- `district_id` — ссылка на официальный район  
- `is_fake` — признак фейкового здания  
- `folk_district_id` — ссылка на народный район  
- `big_folk_district_id` — ссылка на большой народный район  
- `type` — тип здания  
- `city_id` — ссылка на город  
- `coordinates` — географические координаты  

**cities** — справочник городов  
- `id` — уникальный идентификатор  
- `name` — название города  

**districts** — справочник официальных административных районов 
- `id` — уникальный идентификатор  
- `name` — название района  

**folk_districts** — справочник народных/неформальных районов
- `id` — уникальный идентификатор  
- `name` — название района  

**big_folk_districts** — справочник крупных народных районов  
- `id` — уникальный идентификатор  
- `name` — название района  

**streets** — справочник улиц  
- `id` — уникальный идентификатор  
- `name` — название улицы  
- `city_id` — ссылка на город

### Загрузка данных из базы данных

In [2]:
DB_PATH = "Кейс_Аналитика.db"

db_connect = sqlite3.connect(DB_PATH)

In [3]:
blackouts = pd.read_sql("SELECT * FROM blackouts", con=db_connect)

blackouts.head()

,id,start_date,end_date,description,type,initiator_name,source
0,f88cefa506f44ebf8f010b8681b5449e,2018-01-01 00:08:00,2018-01-01 09:00:00,"Авария на сети электроснабжения, ведутся восст...",electricity,МУПВ ВПЭС (электрические сети Ленинского района),Единая дежурная диспетчерская служба города (Л...
1,38ddf6852801fa90cc70f9770239961e,2018-01-01 00:24:00,2018-01-01 10:00:00,"Авария на электролинии, остановка работы насос...",cold_water,МУПВ ВПЭС (электрические сети Ленинского района),Единая дежурная диспетчерская служба города (Л...
2,53c570099fe380dce9e56e2ace9cfa9c,2018-01-01 10:44:00,2018-01-02 18:00:00,Авария в системе водоснабжения дома. Жителям н...,hot_water,"ООО ""Управляющая компания ""Регион-ЖКХ""","Аварийная служба ООО ""Мадикс"""
3,9c1b9ebbd9a698eef046b27cb3568745,2018-01-01 11:32:00,2018-01-01 15:00:00,"Авария на сети электроснабжения, ведутся восст...",electricity,МУПВ ВПЭС (электрические сети Фрунзенского рай...,Единая дежурная диспетчерская служба города (Ф...
4,8aa631cb343aac0731bbde806dfa6d8c,2018-01-01 11:33:00,2018-01-01 15:00:00,"Авария на электролинии, остановка работы насос...",hot_water,МУПВ ВПЭС (электрические сети Фрунзенского рай...,Единая дежурная диспетчерская служба города (Ф...


In [4]:
buildings = pd.read_sql("SELECT * FROM buildings", con=db_connect)

buildings.head()

,id,street_id,number,district_id,is_fake,folk_district_id,big_folk_district_id,type,city_id,coordinates
0,b428b92bb123994a56234bb6eeeed414,37454c5a86ea320f2d5eb4eabdf1ae86,1,504f6b8bd5daeba64beb62bda8b3c12a,0,551a73c4f1b8d1cc72eea5ec752463c4,9fec0dfb8ff6bb1cf6126dc933be3489,нежилое,b428b92bb123994a56234bb6eeeed414,"[{""lat"": 43.14199293270051, ""lon"": 131.9080723..."
1,24aa25303f0f91ede8ee0c0d324c8ee3,37454c5a86ea320f2d5eb4eabdf1ae86,100,87d459096286c57ed5e780205f041e68,0,1297cda65a7da55fd689abff65c09504,28f0c59576ad51c52bf864a76670abc2,жилое многоквартирное,b428b92bb123994a56234bb6eeeed414,"[{""lat"": 43.171246518452826, ""lon"": 131.917043..."
2,504f6b8bd5daeba64beb62bda8b3c12a,37454c5a86ea320f2d5eb4eabdf1ae86,100А,87d459096286c57ed5e780205f041e68,0,1297cda65a7da55fd689abff65c09504,28f0c59576ad51c52bf864a76670abc2,жилое многоквартирное,b428b92bb123994a56234bb6eeeed414,"[{""lat"": 43.17160112621735, ""lon"": 131.9178653..."
3,87d459096286c57ed5e780205f041e68,37454c5a86ea320f2d5eb4eabdf1ae86,100В,87d459096286c57ed5e780205f041e68,0,1297cda65a7da55fd689abff65c09504,28f0c59576ad51c52bf864a76670abc2,жилое многоквартирное,b428b92bb123994a56234bb6eeeed414,"[{""lat"": 43.17208966911542, ""lon"": 131.9183524..."
4,a4f6e19b37548845b4ee10c573809e68,37454c5a86ea320f2d5eb4eabdf1ae86,102,87d459096286c57ed5e780205f041e68,0,1297cda65a7da55fd689abff65c09504,28f0c59576ad51c52bf864a76670abc2,жилое многоквартирное,b428b92bb123994a56234bb6eeeed414,"[{""lat"": 43.172186535721075, ""lon"": 131.917439..."


In [5]:
blackouts_buildings = pd.read_sql("SELECT * FROM blackouts_buildings", con=db_connect)

blackouts_buildings.head()

,blackout_id,building_id
0,None,625c9861f138e4dfeb68cc9b1737d525
1,None,2ef83bdea9aed0734f44337c3ecf9a9a
2,None,cbfc294aceb59a85354aa0648e7e0e47
3,None,98462c9c6a1a96cec4f1e4b8e5c374fa
4,None,31bf19dff7df028f177ac45444b500b8


In [6]:
cities = pd.read_sql("SELECT * FROM cities", con=db_connect)

cities.head()

,id,name
0,24aa25303f0f91ede8ee0c0d324c8ee3,Артем
1,b428b92bb123994a56234bb6eeeed414,Владивосток


In [7]:
districts = pd.read_sql("SELECT * FROM districts", con=db_connect)

districts.head()

,id,name
0,b428b92bb123994a56234bb6eeeed414,Ленинский район
1,24aa25303f0f91ede8ee0c0d324c8ee3,Первомайский район
2,504f6b8bd5daeba64beb62bda8b3c12a,Первореченский район
3,87d459096286c57ed5e780205f041e68,Советский район
4,a4f6e19b37548845b4ee10c573809e68,Фрунзенский район


In [8]:
folk_districts = pd.read_sql("SELECT * FROM folk_districts", con=db_connect)

folk_districts.head()

,id,name
0,b428b92bb123994a56234bb6eeeed414,Центр
1,24aa25303f0f91ede8ee0c0d324c8ee3,БАМ
2,504f6b8bd5daeba64beb62bda8b3c12a,Гризодубова-Сафонова
3,87d459096286c57ed5e780205f041e68,Щитовая
4,a4f6e19b37548845b4ee10c573809e68,Трасса Де-Фриз - Седанка


In [9]:
big_folk_districts = pd.read_sql("SELECT * FROM big_folk_districts", con=db_connect)

big_folk_districts.head()

,id,name
0,b428b92bb123994a56234bb6eeeed414,Центр
1,24aa25303f0f91ede8ee0c0d324c8ee3,БАМ
2,504f6b8bd5daeba64beb62bda8b3c12a,Тихая
3,87d459096286c57ed5e780205f041e68,Шамора
4,a4f6e19b37548845b4ee10c573809e68,Пригород


In [10]:
streets = pd.read_sql("SELECT * FROM streets", con=db_connect)

streets.head()

,id,name,city_id
0,ceba6f5d616341f44f5fcae0425d020b,1-й Байкальский пер.,24aa25303f0f91ede8ee0c0d324c8ee3
1,a7fc0a136f4b1e18a8365d3d29c65c05,1-й Вокзальный пер.,24aa25303f0f91ede8ee0c0d324c8ee3
2,d3ef58b013fa6d05ad5b6cf4210e0fc7,1-й Воровского пер.,24aa25303f0f91ede8ee0c0d324c8ee3
3,4a60bf748c91c074046b86bd48bfa10a,1-й Зареченский пер.,24aa25303f0f91ede8ee0c0d324c8ee3
4,11edbe94c0140fae83c25cd72d11aa0b,1-й Линейный пер.,24aa25303f0f91ede8ee0c0d324c8ee3


### Разведочный анализ данных (EDA)

#### Кол-во строк и типы данных колонок

##### Таблица отключений систем ЖКХ

In [12]:
blackouts.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25264 entries, 0 to 25263
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   id              25264 non-null  object
 1   start_date      25264 non-null  object
 2   end_date        25264 non-null  object
 3   description     25264 non-null  object
 4   type            25264 non-null  object
 5   initiator_name  25264 non-null  object
 6   source          8722 non-null   object
dtypes: object(7)
memory usage: 1.3+ MB


In [11]:
blackouts["start_date"] = pd.to_datetime(blackouts["start_date"])
blackouts["end_date"] = pd.to_datetime(blackouts["end_date"])

Колонки с датами были приведены к типу **datetime**.

##### Справочник зданий

In [14]:
buildings.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 58199 entries, 0 to 58198
Data columns (total 10 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   id                    58199 non-null  object
 1   street_id             58199 non-null  object
 2   number                58199 non-null  object
 3   district_id           34606 non-null  object
 4   is_fake               58199 non-null  int64 
 5   folk_district_id      26009 non-null  object
 6   big_folk_district_id  33063 non-null  object
 7   type                  34631 non-null  object
 8   city_id               58199 non-null  object
 9   coordinates           50521 non-null  object
dtypes: int64(1), object(9)
memory usage: 4.4+ MB


##### Связь отключений со зданиями

In [15]:
blackouts_buildings.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1419556 entries, 0 to 1419555
Data columns (total 2 columns):
 #   Column       Non-Null Count    Dtype 
---  ------       --------------    ----- 
 0   blackout_id  175590 non-null   object
 1   building_id  1419556 non-null  object
dtypes: object(2)
memory usage: 21.7+ MB


##### Справочник городов

In [16]:
cities.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2 entries, 0 to 1
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   id      2 non-null      object
 1   name    2 non-null      object
dtypes: object(2)
memory usage: 160.0+ bytes


##### Справочник официальных административных районов

In [17]:
districts.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   id      5 non-null      object
 1   name    5 non-null      object
dtypes: object(2)
memory usage: 208.0+ bytes


##### Справочник народных/неформальных районов

In [18]:
folk_districts.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3594 entries, 0 to 3593
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   id      3594 non-null   object
 1   name    3594 non-null   object
dtypes: object(2)
memory usage: 56.3+ KB


##### Справочник крупных народных районов

In [19]:
big_folk_districts.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45 entries, 0 to 44
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   id      45 non-null     object
 1   name    45 non-null     object
dtypes: object(2)
memory usage: 848.0+ bytes


##### Справочник улиц

In [20]:
streets.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2023 entries, 0 to 2022
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   id       2023 non-null   object
 1   name     2023 non-null   object
 2   city_id  2023 non-null   object
dtypes: object(3)
memory usage: 47.5+ KB


#### Очистка пустых значений

In [12]:
def print_null_pct(data: pd.DataFrame):
    for col in data.columns:
        null_pct = (data[col].isnull().sum() / len(data) * 100).round(2)
        print(f"{col}: {null_pct}%")

##### Таблица отключений систем ЖКХ

In [22]:
print_null_pct(blackouts)

id: 0.0%
start_date: 0.0%
end_date: 0.0%
description: 0.0%
type: 0.0%
initiator_name: 0.0%
source: 65.48%


In [13]:
blackouts["source"] = blackouts["source"].fillna("Неизвестно")

В таблице с отключениями ЖКХ в столбце **«источник информации»** обнаружено 65% пропусков. Восстановить эти данные не представляется возможным из-за отсутствия точных исходных сведений. Для сохранения целостности данных все пропуски будут заполнены значением **«Неизвестно»**.

##### Справочник зданий

In [24]:
print_null_pct(buildings)

id: 0.0%
street_id: 0.0%
number: 0.0%
district_id: 40.54%
is_fake: 0.0%
folk_district_id: 55.31%
big_folk_district_id: 43.19%
type: 40.5%
city_id: 0.0%
coordinates: 13.19%


In [14]:
real_buildings = buildings[buildings["is_fake"] == 0]

print_null_pct(real_buildings)

id: 0.0%
street_id: 0.0%
number: 0.0%
district_id: 40.85%
is_fake: 0.0%
folk_district_id: 53.5%
big_folk_district_id: 40.95%
type: 35.3%
city_id: 0.0%
coordinates: 0.0%


Пропуски в координатах присутствуют только в строках со зданиями, которые считаются фейками.

In [26]:
tmp_cities = cities.copy().rename(columns={
    "id": "city_id",
    "name": "city_name"
})

tmp_districts = districts.copy().rename(columns={
    "id": "district_id",
    "name": "district_name"
})

tmp_folk_districts = folk_districts.copy().rename(columns={
    "id": "folk_district_id",
    "name": "folk_district_name"
})

tmp_big_folk_districts = big_folk_districts.copy().rename(columns={
    "id": "big_folk_district_id",
    "name": "big_folk_district_name"
})

real_buildings = real_buildings.merge(tmp_cities, on="city_id", how="left")
real_buildings = real_buildings.merge(tmp_districts, on="district_id", how="left")
real_buildings = real_buildings.merge(tmp_folk_districts, on="folk_district_id", how="left")
real_buildings = real_buildings.merge(tmp_big_folk_districts, on="big_folk_district_id", how="left")

In [27]:
real_buildings[["district_name", "folk_district_name", "big_folk_district_name"]] = real_buildings[
    ["district_name", "folk_district_name", "big_folk_district_name"]
].fillna("Неизвестно")

In [28]:
display(HTML(real_buildings.groupby(
    ["city_name", "district_name", "big_folk_district_name", "folk_district_name"]
)["id"].count().reset_index().rename(columns={"id": "count"}).to_html()))

,city_name,district_name,big_folk_district_name,folk_district_name,count
0,Артем,Неизвестно,Неизвестно,Неизвестно,16092
1,Артем,Неизвестно,Неизвестно,Район Хребта Богатая Грива,10
2,Артем,Неизвестно,Неизвестно,Тавайза,1
3,Артем,Неизвестно,Неизвестно,Трудовое,298
4,Артем,Неизвестно,Неизвестно,Угловое,2421
5,Владивосток,Ленинский район,"64, 71 микрорайоны",64 71 микрорайон,2
6,Владивосток,Ленинский район,"64, 71 микрорайоны","64, 71 микрорайон",346
7,Владивосток,Ленинский район,"64, 71 микрорайоны",Неизвестно,181
8,Владивосток,Ленинский район,"64, 71 микрорайоны",ТЭЦ(Золоотвал),1
9,Владивосток,Ленинский район,"64, 71 микрорайоны",Фадеева,16


Для восстановления пропущенных значений районов у зданий наиболее точным решением является обучение ML-модели, которая будет предсказывать район по географическим координатам. Этот подход использует существующие данные без привлечения внешних источников. Но реализовывать это будем потом.

##### Связь отключений со зданиями

In [29]:
print_null_pct(blackouts_buildings)

blackout_id: 87.63%
building_id: 0.0%


In [15]:
blackouts_buildings = blackouts_buildings.dropna()

##### Справочник городов

In [31]:
print_null_pct(cities)

id: 0.0%
name: 0.0%


##### Справочник официальных административных районов

In [32]:
print_null_pct(districts)

id: 0.0%
name: 0.0%


##### Справочник народных/неформальных районов

In [33]:
print_null_pct(folk_districts)

id: 0.0%
name: 0.0%


##### Справочник крупных народных районов

In [34]:
print_null_pct(big_folk_districts)

id: 0.0%
name: 0.0%


##### Справочник улиц

In [35]:
print_null_pct(streets)

id: 0.0%
name: 0.0%
city_id: 0.0%


### Предобработка данных (преобразование существующих, добавление новых)

#### Разбиение колонки координат на отдельные колонки (широта, долгота)

In [16]:
def separate_coordinates(coordinates: Any) -> pd.Series:
    if coordinates:
        regex = "-*\d+.\d+"
        separated_coordinates = re.findall(regex, coordinates)

        if len(separated_coordinates) != 2:
            return pd.Series([None, None])
        return pd.Series(separated_coordinates).astype(np.float64)
    
    return pd.Series([None, None])

In [17]:
buildings[["latitude", "longitude"]] = buildings["coordinates"].progress_apply(separate_coordinates)

100%|██████████| 58199/58199 [00:05<00:00, 11515.16it/s]


In [38]:
buildings.head()

,id,street_id,number,district_id,is_fake,folk_district_id,big_folk_district_id,type,city_id,coordinates,latitude,longitude
0,b428b92bb123994a56234bb6eeeed414,37454c5a86ea320f2d5eb4eabdf1ae86,1,504f6b8bd5daeba64beb62bda8b3c12a,0,551a73c4f1b8d1cc72eea5ec752463c4,9fec0dfb8ff6bb1cf6126dc933be3489,нежилое,b428b92bb123994a56234bb6eeeed414,"[{""lat"": 43.14199293270051, ""lon"": 131.9080723...",43.141993,131.908072
1,24aa25303f0f91ede8ee0c0d324c8ee3,37454c5a86ea320f2d5eb4eabdf1ae86,100,87d459096286c57ed5e780205f041e68,0,1297cda65a7da55fd689abff65c09504,28f0c59576ad51c52bf864a76670abc2,жилое многоквартирное,b428b92bb123994a56234bb6eeeed414,"[{""lat"": 43.171246518452826, ""lon"": 131.917043...",43.171247,131.917043
2,504f6b8bd5daeba64beb62bda8b3c12a,37454c5a86ea320f2d5eb4eabdf1ae86,100А,87d459096286c57ed5e780205f041e68,0,1297cda65a7da55fd689abff65c09504,28f0c59576ad51c52bf864a76670abc2,жилое многоквартирное,b428b92bb123994a56234bb6eeeed414,"[{""lat"": 43.17160112621735, ""lon"": 131.9178653...",43.171601,131.917865
3,87d459096286c57ed5e780205f041e68,37454c5a86ea320f2d5eb4eabdf1ae86,100В,87d459096286c57ed5e780205f041e68,0,1297cda65a7da55fd689abff65c09504,28f0c59576ad51c52bf864a76670abc2,жилое многоквартирное,b428b92bb123994a56234bb6eeeed414,"[{""lat"": 43.17208966911542, ""lon"": 131.9183524...",43.172090,131.918352
4,a4f6e19b37548845b4ee10c573809e68,37454c5a86ea320f2d5eb4eabdf1ae86,102,87d459096286c57ed5e780205f041e68,0,1297cda65a7da55fd689abff65c09504,28f0c59576ad51c52bf864a76670abc2,жилое многоквартирное,b428b92bb123994a56234bb6eeeed414,"[{""lat"": 43.172186535721075, ""lon"": 131.917439...",43.172187,131.917440


#### Предсказание района по координатам

In [18]:
buildings_with_coords = buildings[buildings["latitude"].notna()]

In [19]:
buildings_for_model = buildings_with_coords[buildings_with_coords[[
    "district_id", "folk_district_id", "big_folk_district_id"
]].notna().all(axis=1)]

Создан датасет, на котором будет проходить обучение модели для предсказания района по координатам

In [20]:
buildings_for_model["target_district"] = buildings_for_model["district_id"] + "_" + \
      buildings_for_model["folk_district_id"] + "_" + buildings_for_model["big_folk_district_id"]

In [ ]:
target_districts = buildings_for_model.groupby("target_district")[
    ["id"]].count().rename(columns={"id": "count"})
target_districts = target_districts[target_districts["count"] > 1].index # Отбираем целевые признаки, где хотя бы 2 примера

buildings_for_model = buildings_for_model[buildings_for_model["target_district"].isin(target_districts)]

Мы формируем целевую переменную как комбинацию трех идентификаторов районов в формате **[district_id]\_[folk_district_id]\_[big_folk_district_id]**. Модель, обученная на координатах, будет предсказывать эту комбинированную метку для зданий с пропущенными районами, после чего результат разделяется на три исходных признака.

In [29]:
X = buildings_for_model[["latitude", "longitude"]]
y = buildings_for_model["target_district"]

district_encoder = LabelEncoder()
y = district_encoder.fit_transform(y)

In [32]:
models_param_grid = [
    {
        'name': 'RandomForest',
        'model': RandomForestClassifier(random_state=42, n_jobs=-1),
        'param_grid': {
            'n_estimators': [100, 200, 300],
            'max_depth': [10, 15, 20, None],
            'min_samples_split': [2, 5, 10],
            'min_samples_leaf': [1, 2, 4],
            'class_weight': [None, 'balanced']
        }
    },
    {
        'name': 'XGBoost',
        'model': XGBClassifier(random_state=42, n_jobs=-1),
        'param_grid': {
            'n_estimators': [100, 200, 300],
            'max_depth': [3, 6, 9],
            'learning_rate': [0.01, 0.1, 0.2],
            'subsample': [0.8, 0.9, 1.0],
            'colsample_bytree': [0.8, 0.9, 1.0]
        }
    },
    {
        'name': 'LightGBM', 
        'model': LGBMClassifier(random_state=42, n_jobs=-1, verbose=-1),
        'param_grid': {
            'n_estimators': [100, 200, 300],
            'max_depth': [5, 10, 15],
            'learning_rate': [0.01, 0.05, 0.1],
            'num_leaves': [31, 63, 127],
            'subsample': [0.8, 0.9, 1.0]
        }
    },
    {
        'name': 'GradientBoosting',
        'model': GradientBoostingClassifier(random_state=42),
        'param_grid': {
            'n_estimators': [100, 200],
            'max_depth': [3, 5, 7],
            'learning_rate': [0.05, 0.1, 0.2],
            'subsample': [0.8, 0.9, 1.0]
        }
    }
]

In [33]:
f1_weighted_scorer = make_scorer(f1_score, average='weighted')

best_score = 0
district_predictor = None
district_predictor_name = ""

stratified_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print("--- Сравнение моделей ---")

for model_config in models_param_grid:
    print(f"Обучение {model_config['name']}")

    try:
        grid_search = GridSearchCV(
            estimator=model_config['model'],
            param_grid=model_config['param_grid'],
            scoring=f1_weighted_scorer,
            cv=stratified_cv,
            n_jobs=-1,
            verbose=1,
            return_train_score=True
        )
        
        grid_search.fit(X, y)
        
        print(f"Лучший F1 Weighted Score для {model_config['name']}: {grid_search.best_score_:.4f}")
        print(f"Лучшие параметры: {grid_search.best_params_}")
        
        if grid_search.best_score_ > best_score:
            best_score = grid_search.best_score_
            district_predictor = grid_search.best_estimator_
            district_predictor_name = model_config['name']
    
    except Exception as e:
        print(f"Ошибка при обучении {model_config['name']}: {e}")

print(f"Лучшая модель: {district_predictor_name}")
print(f"Лучший F1 Weighted Score: {best_score:.4f}")

--- Сравнение моделей ---
Обучение RandomForest
Fitting 5 folds for each of 216 candidates, totalling 1080 fits
Лучший F1 Weighted Score для RandomForest: 0.9704
Лучшие параметры: {'class_weight': None, 'max_depth': None, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 200}
Обучение XGBoost
Fitting 5 folds for each of 243 candidates, totalling 1215 fits


KeyboardInterrupt: 

In [ ]:
cv_scores_weighted = []
cv_scores_macro = []

for fold, (train_idx, val_idx) in enumerate(stratified_cv.split(X), 1):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y[train_idx], y[val_idx]
    
    district_predictor.fit(X_train, y_train)
    y_pred = district_predictor.predict(X_val)
    
    f1_weighted = f1_score(y_val, y_pred, average='weighted')
    f1_macro = f1_score(y_val, y_pred, average='macro')
    cv_scores_weighted.append(f1_weighted)
    cv_scores_macro.append(f1_macro)
    print(f"Fold {fold}: F1 Weighted = {f1_weighted:.4f}, F1 Macro = {f1_macro:.4f}")

print(f"Средний F1 Weighted: {np.mean(cv_scores_weighted):.4f} (+/- {np.std(cv_scores_weighted):.4f})")
print(f"Средний F1 Macro: {np.mean(cv_scores_macro):.4f} (+/- {np.std(cv_scores_macro):.4f})")

Fold 1: F1 Weighted = 0.9661, F1 Macro = 0.7579
Fold 2: F1 Weighted = 0.9675, F1 Macro = 0.7466
Fold 3: F1 Weighted = 0.9693, F1 Macro = 0.7533
Fold 4: F1 Weighted = 0.9648, F1 Macro = 0.7774
Fold 5: F1 Weighted = 0.9598, F1 Macro = 0.7342
Средний F1 Weighted: 0.9655 (+/- 0.0032)
Средний F1 Macro: 0.7539 (+/- 0.0142)


Модель предсказания районов по координатам показывает высокую эффективность: F1 Weighted 0.965 при стабильности across фолдов. Разрыв между F1 Weighted и F1 Macro (0.754) указывает на дисбаланс классов - модель хуже предсказывает малые районы (так как представлено либо мало примеров, либо они очень близки к другим микрорайонам).

Решение готово к внедрению для автоматического заполнения пропусков в данных.

In [ ]:
with open("District_Predictor.sav", "wb") as f:
    pickle.dump(district_predictor, f)

In [ ]:
with open("District_Predictor.sav", "rb") as f:
    district_predictor = pickle.load(f)

In [ ]:
def fill_district(row: pd.Series) -> pd.Series:
    district_id = row.district_id
    folk_district_id = row.folk_district_id
    big_folk_district_id = row.big_folk_district_id
    latitude = row.latitude
    longitude = row.longitude

    if district_id is None or folk_district_id is None or big_folk_district_id is None:
        if latitude and longitude:
            prediction = district_predictor.predict([[latitude, longitude]])
            prediction = district_encoder.inverse_transform(prediction)[0]
            district_id, folk_district_id, big_folk_district_id = prediction.split("_")
        else:
            pd.Series([None, None, None, None, None])
        
    return pd.Series([district_id, folk_district_id, big_folk_district_id, latitude, longitude])

: 

In [ ]:
district_cols = ["district_id", "folk_district_id", "big_folk_district_id", "latitude", "longitude"]

buildings[district_cols] = buildings[district_cols].progress_apply(fill_district, axis=1)

  0%|          | 0/58199 [00:00<?, ?it/s]

100%|██████████| 58199/58199 [17:21<00:00, 55.89it/s]  


#### Предобработка признака - тип постройки

In [ ]:
buildings.groupby("type")["id"].count()

type
Жилое                          433
Жилое сельское                 576
Жилое сельское строение        291
Жилое строение                  87
Нежилое                        204
Нежилое строение                76
Постройка без адреса             3
Производственное                27
Производственное строение        1
Разрушенное                      3
СНТ                            156
Строение жилое                   4
Строение жилое сельское          3
Строение нежилое                 3
Строение производственное        1
Строение строящееся              2
Строящееся                      63
Строящееся строение             15
гаражи                          12
дача                           914
жилое многоквартирное         4836
нежилое                       5328
общественное                   337
планируемое                     39
производственное               290
разрушенное                     91
строение                        55
строящееся                     840
частный дом    

In [ ]:
group_mapping = {
    # Жилые объекты
    "жилое": "Жилые объекты",
    "жилое строение": "Жилые объекты",
    "жилое многоквартирное": "Многоквартирные дома",
    "строение жилое": "Жилые объекты",

    # Сельские жилые
    "жилое сельское": "Сельские жилые объекты",
    "жилое сельское строение": "Сельские жилые объекты",
    "строение жилое сельское": "Сельские жилые объекты",

    # Частные дома, дачи, СНТ
    "частный дом": "Частные дома",
    "дача": "Дачи",
    "снт": "Садовые и дачные участки",

    # Нежилые
    "нежилое": "Нежилые объекты",
    "нежилое строение": "Нежилые объекты",
    "строение нежилое": "Нежилые объекты",
    "общественное": "Общественные объекты",

    # Производственные
    "производственное": "Производственные объекты",
    "производственное строение": "Производственные объекты",
    "строение производственное": "Производственные объекты",

    # Строящиеся / планируемые
    "строящееся": "Строящиеся объекты",
    "строящееся строение": "Строящиеся объекты",
    "строение строящееся": "Строящиеся объекты",
    "планируемое": "Планируемые объекты",

    # Разрушенные / без адреса
    "разрушенное": "Разрушенные объекты",
    "постройка без адреса": "Безадресные постройки",

    # Прочие
    "строение": "Прочие постройки",
    "гаражи": "Гаражи"
}


In [ ]:
def prepare_building_type(building_type: str):
    if building_type is None:
        return None
    else:
        building_type = building_type.lower()
        return group_mapping.get(building_type, None)

In [ ]:
buildings["type"] = buildings["type"].progress_apply(prepare_building_type)

100%|██████████| 58199/58199 [00:00<00:00, 1663504.40it/s]


In [ ]:
buildings.groupby("type")["id"].count()

type
Безадресные постройки           3
Гаражи                         12
Дачи                          914
Жилые объекты                 524
Многоквартирные дома         4836
Нежилые объекты              5611
Общественные объекты          337
Планируемые объекты            39
Производственные объекты      319
Прочие постройки               55
Разрушенные объекты            94
Садовые и дачные участки      156
Сельские жилые объекты        870
Строящиеся объекты            920
Частные дома                19941
Name: id, dtype: int64

Типы зданий сгруппированы по общим характеристикам. Для восстановления пропущенных значений будет использована модель машинного обучения, обученная на существующих данных.

In [ ]:
buildings_with_type = buildings[(buildings["type"].notna()) & (buildings["latitude"].notna())]

print_null_pct(buildings_with_type)

id: 0.0%
street_id: 0.0%
number: 0.0%
district_id: 0.0%
is_fake: 0.0%
folk_district_id: 0.0%
big_folk_district_id: 0.0%
type: 0.0%
city_id: 0.0%
coordinates: 0.0%
latitude: 0.0%
longitude: 0.0%


In [ ]:
X = buildings_with_type[["latitude", "longitude"]]
y = buildings_with_type["type"]

type_encoder = LabelEncoder()
y = type_encoder.fit_transform(y)

In [67]:
best_score = 0
building_type_predictor = None
building_type_predictor_name = ""

stratified_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print("--- Сравнение моделей ---")

for model_config in models_param_grid:
    print(f"Обучение {model_config['name']}")
    
    grid_search = GridSearchCV(
        estimator=model_config['model'],
        param_grid=model_config['param_grid'],
        scoring=f1_weighted_scorer,
        cv=stratified_cv,
        n_jobs=-1,
        verbose=1,
        return_train_score=True
    )
    
    grid_search.fit(X, y)
    
    print(f"Лучший F1 Weighted Score для {model_config['name']}: {grid_search.best_score_:.4f}")
    print(f"Лучшие параметры: {grid_search.best_params_}")
    
    if grid_search.best_score_ > best_score:
        best_score = grid_search.best_score_
        building_type_predictor = grid_search.best_estimator_
        building_type_predictor_name = model_config['name']

print(f"Лучшая модель: {building_type_predictor_name}")
print(f"Лучший F1 Weighted Score: {best_score:.4f}")

--- Сравнение моделей ---
Обучение XGBoost
Fitting 5 folds for each of 243 candidates, totalling 1215 fits


ValueError: 
All the 1215 fits failed.
It is very likely that your model is misconfigured.
You can try to debug the error by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
1215 fits failed with the following error:
Traceback (most recent call last):
  File "C:\Users\Руслан\AppData\Roaming\Python\Python310\site-packages\sklearn\model_selection\_validation.py", line 859, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "c:\ProgramData\miniconda3\envs\tensorflow_env\lib\site-packages\xgboost\core.py", line 726, in inner_f
    return func(**kwargs)
  File "c:\ProgramData\miniconda3\envs\tensorflow_env\lib\site-packages\xgboost\sklearn.py", line 1512, in fit
    train_dmatrix, evals = _wrap_evaluation_matrices(
  File "c:\ProgramData\miniconda3\envs\tensorflow_env\lib\site-packages\xgboost\sklearn.py", line 596, in _wrap_evaluation_matrices
    train_dmatrix =

In [53]:
f1_weighted_scorer = make_scorer(f1_score, average='weighted')

# Параметры для GridSearch
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [5, 10, 15, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2', None]
}

# Настройка GridSearchCV
grid_search = GridSearchCV(
    estimator=RandomForestClassifier(random_state=42, n_jobs=-1),
    param_grid=param_grid,
    scoring=f1_weighted_scorer,
    cv=5,
    n_jobs=-1,
    verbose=1,
    return_train_score=True
)

grid_search.fit(X, y)

print(f"Лучший F1 Weighted Score: {grid_search.best_score_:.4f}")

Fitting 5 folds for each of 324 candidates, totalling 1620 fits
Лучший F1 Weighted Score: 0.6631


In [54]:
building_type_predictor = grid_search.best_estimator_

cv_scores_weighted = []
cv_scores_macro = []
kf = KFold(n_splits=5, shuffle=True, random_state=42)

for fold, (train_idx, val_idx) in enumerate(kf.split(X), 1):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y[train_idx], y[val_idx]
    
    building_type_predictor.fit(X_train, y_train)
    y_pred = building_type_predictor.predict(X_val)
    
    f1_weighted = f1_score(y_val, y_pred, average='weighted')
    f1_macro = f1_score(y_val, y_pred, average='macro')
    cv_scores_weighted.append(f1_weighted)
    cv_scores_macro.append(f1_macro)
    print(f"Fold {fold}: F1 Weighted = {f1_weighted:.4f}, F1 Macro = {f1_macro:.4f}")

print(f"Средний F1 Weighted: {np.mean(cv_scores_weighted):.4f} (+/- {np.std(cv_scores_weighted):.4f})")
print(f"Средний F1 Macro: {np.mean(cv_scores_macro):.4f} (+/- {np.std(cv_scores_macro):.4f})")

Fold 1: F1 Weighted = 0.7953, F1 Macro = 0.2874
Fold 2: F1 Weighted = 0.7889, F1 Macro = 0.2728
Fold 3: F1 Weighted = 0.7873, F1 Macro = 0.2613
Fold 4: F1 Weighted = 0.7858, F1 Macro = 0.2837
Fold 5: F1 Weighted = 0.8044, F1 Macro = 0.2731
Средний F1 Weighted: 0.7924 (+/- 0.0068)
Средний F1 Macro: 0.2756 (+/- 0.0092)


In [ ]:
f1_weighted_scorer = make_scorer(f1_score, average='weighted')

# Параметры для GridSearch
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [5, 10, 15, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2', None]
}

# Настройка GridSearchCV
grid_search = GridSearchCV(
    estimator=RandomForestClassifier(random_state=42, n_jobs=-1),
    param_grid=param_grid,
    scoring=f1_weighted_scorer,
    cv=5,
    n_jobs=-1,
    verbose=1,
    return_train_score=True
)

grid_search.fit(X, y)

print(f"Лучший F1 Weighted Score: {grid_search.best_score_:.4f}")